# Fine-tuning PanEcho for Ejection Fraction (EF) Prediction

This notebook fine-tunes the pretrained PanEcho model to focus specifically on predicting ejection fraction (EF) from echocardiogram videos.

**What is EF?** Ejection fraction is the percentage of blood pumped out of the left ventricle with each heartbeat. Normal is ≥55%. It's one of the most important measures of heart function.

**What is fine-tuning?** PanEcho was already trained to predict 39+ cardiac measurements. We're going to take that pretrained model and further train it specifically on EF so it gets even better at that one task.

---
**Before running:** make sure you have activated the panecho conda environment and launched Jupyter from inside it.

## Cell 1 — Imports

We load all the libraries we need up front:
- `torch` — the deep learning framework everything runs on
- `cv2` — reads video files frame by frame
- `pandas` — reads the FileList.csv spreadsheet
- `numpy` — handles arrays of pixel values
- `matplotlib` — plots training curves at the end
- `tqdm` — shows a progress bar during training

In [ ]:
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

print("All imports OK")
print(f"PyTorch version: {torch.__version__}")

## Cell 2 — Configuration

Set your paths and hyperparameters here. This is the only cell you need to edit to point at your data.

**Hyperparameters** are settings that control how training works — not learned by the model, but chosen by you:
- `CLIP_LEN` — how many frames to sample from each video
- `BATCH_SIZE` — how many videos to process at once before updating weights
- `PHASE1_EPOCHS` / `PHASE2_EPOCHS` — how many full passes through the data
- `LR_PHASE1` / `LR_PHASE2` — learning rate: how big a step to take when updating weights

In [ ]:
# ── Edit this path to point at your EchoNet-Dynamic folder ──
DATA_DIR  = Path.home() / "PanEchoTest" / "data" / "EchoNet-Dynamic"
VIDEO_DIR = DATA_DIR / "Videos"
OUT_DIR   = Path("ef_finetune")
OUT_DIR.mkdir(exist_ok=True)

# ── Hyperparameters ──
CLIP_LEN       = 16      # frames sampled per video
BATCH_SIZE     = 4       # videos per training step
PHASE1_EPOCHS  = 5       # epochs with backbone frozen
PHASE2_EPOCHS  = 10      # epochs with full model unfrozen
LR_PHASE1      = 1e-3    # learning rate for phase 1
LR_PHASE2      = 1e-4    # learning rate for phase 2 (lower = gentler updates)

# ── Verify paths exist ──
assert DATA_DIR.exists(),  f"Data directory not found: {DATA_DIR}"
assert VIDEO_DIR.exists(), f"Videos directory not found: {VIDEO_DIR}"
print(f"Data dir:  {DATA_DIR}")
print(f"Video dir: {VIDEO_DIR}")
print(f"Output:    {OUT_DIR}")

## Cell 3 — Pick a device

The model runs on a **device** — either your CPU (slow) or a GPU (fast):
- `mps` — Apple Silicon GPU (M1/M2/M3/M4), ~3-5x faster than CPU
- `cuda` — NVIDIA GPU
- `cpu` — fallback, works everywhere but slow for training

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

## Cell 4 — Video loading function

This function takes a path to an `.avi` file and returns a tensor (a multi-dimensional array of numbers) the model can process.

Steps:
1. Read all frames from the video
2. Uniformly pick `CLIP_LEN` frames (e.g. evenly spaced across the video)
3. Resize each frame to 224×224 pixels
4. Normalize pixel values using ImageNet statistics — this is required because PanEcho was pretrained expecting this exact scale

In [ ]:
# ImageNet normalization constants (must match what PanEcho was trained with)
_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None, None]
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None, None]

def load_video(path: Path, clip_len: int, size: int = 224):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return None

    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)          # OpenCV uses BGR, convert to RGB
        frame = cv2.resize(frame, (size, size))                  # resize to 224x224
        frames.append(frame)
    cap.release()

    if not frames:
        return None

    # Uniformly sample clip_len frames
    if len(frames) >= clip_len:
        idx = np.linspace(0, len(frames) - 1, clip_len, dtype=int)
        frames = [frames[i] for i in idx]
    else:
        while len(frames) < clip_len:       # pad short videos by repeating last frame
            frames.append(frames[-1])

    # Stack into (C, T, H, W) and normalize
    video = np.stack(frames).transpose(3, 0, 1, 2).astype(np.float32) / 255.0
    video = (video - _MEAN) / _STD
    return torch.from_numpy(video)   # shape: (3, clip_len, 224, 224)

print("load_video function defined")

## Cell 5 — Inspect a video

Before training, let's load one video and see what it looks like. This is a good sanity check to make sure the loading function works correctly.

In [ ]:
# Load the FileList to get a sample filename
filelist = pd.read_csv(DATA_DIR / "FileList.csv")
print(f"Total videos in dataset: {len(filelist)}")
print(f"\nFirst 5 rows:")
filelist.head()

In [ ]:
# Load one video and display a few frames
sample_fname = str(filelist.iloc[0]["FileName"])
sample_path  = VIDEO_DIR / (sample_fname if sample_fname.endswith(".avi") else sample_fname + ".avi")

tensor = load_video(sample_path, CLIP_LEN)
print(f"Video tensor shape: {tensor.shape}")
print(f"  → (channels=3, frames={CLIP_LEN}, height=224, width=224)")
print(f"Ground truth EF: {filelist.iloc[0]['EF']:.1f}%")

# Show 4 evenly spaced frames
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
frame_indices = [0, 5, 10, 15]
for ax, fi in zip(axes, frame_indices):
    # Undo normalization for display
    frame = tensor[:, fi, :, :].numpy().transpose(1, 2, 0)
    frame = frame * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    frame = np.clip(frame, 0, 1)
    ax.imshow(frame, cmap="gray")
    ax.set_title(f"Frame {fi}")
    ax.axis("off")
plt.suptitle(f"Sample echo: {sample_fname}  (GT EF = {filelist.iloc[0]['EF']:.1f}%)", y=1.02)
plt.tight_layout()
plt.show()

## Cell 6 — Dataset class

PyTorch needs a **Dataset** object — a class that knows how to return one (video, EF) pair at a time. The training loop will call `__getitem__` repeatedly to feed videos into the model.

Think of it like a filing cabinet: the DataLoader pulls out one folder at a time and hands it to the model.

In [ ]:
class EchoEFDataset(Dataset):
    def __init__(self, filelist: pd.DataFrame, video_dir: Path, clip_len: int):
        # Drop any rows where EF is missing
        self.rows      = filelist.dropna(subset=["EF"]).reset_index(drop=True)
        self.video_dir = video_dir
        self.clip_len  = clip_len

    def __len__(self):
        return len(self.rows)   # total number of videos

    def __getitem__(self, idx):
        row   = self.rows.iloc[idx]
        fname = str(row["FileName"])
        avi   = fname if fname.endswith(".avi") else fname + ".avi"

        tensor = load_video(self.video_dir / avi, self.clip_len)
        if tensor is None:                                          # if video can't load, return zeros
            tensor = torch.zeros(3, self.clip_len, 224, 224)

        ef = torch.tensor(float(row["EF"]), dtype=torch.float32)   # ground truth EF as a number
        return tensor, ef


# Split into train / val using the Split column in FileList.csv
train_df = filelist[filelist["Split"].str.upper() == "TRAIN"]
val_df   = filelist[filelist["Split"].str.upper() == "VAL"]

train_ds = EchoEFDataset(train_df, VIDEO_DIR, CLIP_LEN)
val_ds   = EchoEFDataset(val_df,   VIDEO_DIR, CLIP_LEN)

# DataLoader batches videos together for efficient processing
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training videos:   {len(train_ds)}")
print(f"Validation videos: {len(val_ds)}")
print(f"EF range in train: {train_df['EF'].min():.1f}% – {train_df['EF'].max():.1f}%")
print(f"Mean EF in train:  {train_df['EF'].mean():.1f}%")

## Cell 7 — Load PanEcho

We download the pretrained PanEcho model from CarDS-Yale's GitHub via PyTorch Hub. On first run this downloads ~300 MB of weights. After that it's cached locally.

The model already knows a lot about hearts — we're going to build on top of that knowledge rather than starting from scratch.

In [ ]:
print("Loading PanEcho (downloads weights on first run, ~300 MB) …")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model = torch.hub.load(
        "CarDS-Yale/PanEcho",
        "PanEcho",
        force_reload=False,
        clip_len=CLIP_LEN,
    )
model = model.to(DEVICE)
print("Model loaded and moved to", DEVICE)

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## Cell 8 — Test one forward pass

Before training, confirm the model runs and outputs an EF prediction. A **forward pass** means feeding one video through the model and getting predictions back — no learning yet, just checking it works.

In [ ]:
# Grab one batch from the validation loader
sample_videos, sample_efs = next(iter(val_loader))

with torch.no_grad():
    preds = model(sample_videos.to(DEVICE))

print("Model output keys:", list(preds.keys()))
print(f"\nEF predictions:   {preds['EF'].squeeze().cpu().tolist()}")
print(f"EF ground truths: {sample_efs.tolist()}")

## Cell 9 — Metrics function

After each epoch we measure how well the model is doing using three numbers:

- **MAE** (Mean Absolute Error) — average prediction error in percentage points. If MAE = 5, the model is off by 5% EF on average.
- **RMSE** (Root Mean Square Error) — similar to MAE but penalizes big mistakes more heavily.
- **R²** — how much of the variation in EF the model explains. 1.0 = perfect, 0.0 = no better than guessing the mean.

In [ ]:
def compute_metrics(preds: torch.Tensor, targets: torch.Tensor) -> dict:
    mae    = (preds - targets).abs().mean().item()
    rmse   = ((preds - targets) ** 2).mean().sqrt().item()
    ss_res = ((targets - preds) ** 2).sum()
    ss_tot = ((targets - targets.mean()) ** 2).sum()
    r2     = (1 - ss_res / ss_tot).item()
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

print("compute_metrics defined")

## Cell 9a — Baseline: measure EF error BEFORE fine-tuning

Before changing anything, we evaluate the pretrained PanEcho weights on the validation set.
This is the **baseline** — the starting EF error we want to improve on.
We save these numbers so we can compare them after fine-tuning.

In [ ]:
# Evaluate pretrained (unmodified) PanEcho on the validation set
model.eval()
baseline_preds, baseline_targets = [], []

with torch.no_grad():
    for videos, efs in tqdm(val_loader, desc="baseline eval"):
        preds_dict = model(videos.to(DEVICE))
        baseline_preds.append(preds_dict["EF"].squeeze().cpu())
        baseline_targets.append(efs)

baseline_preds   = torch.cat(baseline_preds)
baseline_targets = torch.cat(baseline_targets)
baseline_metrics = compute_metrics(baseline_preds, baseline_targets)

print("Pretrained PanEcho — validation set (BEFORE fine-tuning):")
print(f"  MAE:  {baseline_metrics['MAE']:.2f}%")
print(f"  RMSE: {baseline_metrics['RMSE']:.2f}%")
print(f"  R²:   {baseline_metrics['R2']:.3f}")

## Cell 10 — Training loop function

This function runs one epoch (one full pass through all videos). It works for both training and validation:

- **Training:** feeds each video through the model, computes the error (**loss**), and updates the model weights to reduce that error
- **Validation:** feeds videos through but does NOT update weights — just measures how well the current model performs on unseen data

**HuberLoss** is used instead of plain MSE because it's less sensitive to outlier EF values (e.g. a very unusual EF of 10% won't dominate the loss).

In [ ]:
def run_epoch(model, loader, optimizer, device, train: bool):
    model.train(train)          # train=True enables dropout etc; train=False disables them
    all_preds, all_targets = [], []
    total_loss = 0.0
    loss_fn = nn.HuberLoss()    # smooth version of MAE, robust to outliers

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for videos, efs in tqdm(loader, desc="train" if train else "val ", leave=False):
            videos = videos.to(device)
            efs    = efs.to(device)

            # Forward pass: run videos through model, extract EF prediction
            preds_dict = model(videos)
            ef_pred    = preds_dict["EF"].squeeze()

            # Compute how wrong the prediction is
            loss = loss_fn(ef_pred, efs)

            if train:
                optimizer.zero_grad()                                    # clear previous gradients
                loss.backward()                                          # compute gradients
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent exploding gradients
                optimizer.step()                                         # update weights

            total_loss += loss.item() * len(efs)
            all_preds.append(ef_pred.detach().cpu())
            all_targets.append(efs.cpu())

    all_preds   = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    metrics = compute_metrics(all_preds, all_targets)
    metrics["loss"] = total_loss / len(all_targets)
    return metrics

print("run_epoch defined")

## Cell 11 — Phase 1: train EF head only (backbone frozen)

In Phase 1 we **freeze** most of the model — those weights are locked and won't change. We only allow the EF-specific output layer to update.

**Why?** The backbone (the video encoder) already learned powerful features from pretraining. If we immediately train everything with a high learning rate, we risk "forgetting" those features. Warming up the EF head first is safer and faster.

The **learning rate scheduler** gradually reduces the learning rate over the epochs following a cosine curve — big steps early, tiny steps at the end to settle into a good solution.

In [ ]:
print("── Phase 1: head-only training ──")

# Freeze everything, then unfreeze EF-related parameters
for name, p in model.named_parameters():
    p.requires_grad = "ef" in name.lower() or "head" in name.lower()

trainable = [p for p in model.parameters() if p.requires_grad]
if not trainable:
    print("No EF-specific params found — unfreezing last 20 parameters instead.")
    for p in list(model.parameters())[-20:]:
        p.requires_grad = True
    trainable = [p for p in model.parameters() if p.requires_grad]

trainable_count = sum(p.numel() for p in trainable)
print(f"Trainable parameters: {trainable_count:,} (frozen: {total_params - trainable_count:,})")

optimizer = torch.optim.AdamW(trainable, lr=LR_PHASE1, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS)

best_val_mae = float("inf")
log_rows = []

for epoch in range(1, PHASE1_EPOCHS + 1):
    tr = run_epoch(model, train_loader, optimizer, DEVICE, train=True)
    vl = run_epoch(model, val_loader,   optimizer, DEVICE, train=False)
    scheduler.step()

    log_rows.append({"phase": 1, "epoch": epoch,
                     **{f"train_{k}": v for k, v in tr.items()},
                     **{f"val_{k}": v for k, v in vl.items()}})

    print(f"[P1 epoch {epoch:02d}]  train MAE={tr['MAE']:.2f}  val MAE={vl['MAE']:.2f}  R²={vl['R2']:.3f}")

    if vl["MAE"] < best_val_mae:
        best_val_mae = vl["MAE"]
        torch.save(model.state_dict(), OUT_DIR / "best_ef_model.pt")
        print(f"           ↳ New best model saved (val MAE={best_val_mae:.2f})")

## Cell 12 — Phase 2: full fine-tuning (all layers unfrozen)

Now we unfreeze the entire model and train everything together at a much lower learning rate (`1e-4` vs `1e-3`). The low rate means the backbone updates very gently — it refines rather than rewrites what it already knows.

This phase takes longer but usually gives the best final accuracy.

In [ ]:
print("── Phase 2: full fine-tuning ──")

# Unfreeze everything
for p in model.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PHASE2, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)

for epoch in range(1, PHASE2_EPOCHS + 1):
    tr = run_epoch(model, train_loader, optimizer, DEVICE, train=True)
    vl = run_epoch(model, val_loader,   optimizer, DEVICE, train=False)
    scheduler.step()

    log_rows.append({"phase": 2, "epoch": epoch,
                     **{f"train_{k}": v for k, v in tr.items()},
                     **{f"val_{k}": v for k, v in vl.items()}})

    print(f"[P2 epoch {epoch:02d}]  train MAE={tr['MAE']:.2f}  val MAE={vl['MAE']:.2f}  R²={vl['R2']:.3f}")

    if vl["MAE"] < best_val_mae:
        best_val_mae = vl["MAE"]
        torch.save(model.state_dict(), OUT_DIR / "best_ef_model.pt")
        print(f"           ↳ New best model saved (val MAE={best_val_mae:.2f})")

# Save the training log
log_df = pd.DataFrame(log_rows)
log_df.to_csv(OUT_DIR / "training_log.csv", index=False)
print(f"\nBest val MAE across all epochs: {best_val_mae:.2f}%")
print(f"Saved to: {OUT_DIR}")

## Cell 13 — Plot training curves

Visualize how MAE and R² changed over every epoch. Good training looks like:
- MAE steadily **decreasing**
- R² steadily **increasing**
- Train and val curves staying close together (if val is much worse, the model is overfitting)

In [ ]:
log_df = pd.read_csv(OUT_DIR / "training_log.csv")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for metric, ax in zip(["MAE", "RMSE", "R2"], axes):
    ax.plot(log_df[f"train_{metric}"], label="Train", marker="o")
    ax.plot(log_df[f"val_{metric}"],   label="Val",   marker="o")
    # Mark the boundary between Phase 1 and Phase 2
    if PHASE1_EPOCHS > 0 and PHASE2_EPOCHS > 0:
        ax.axvline(x=PHASE1_EPOCHS - 0.5, color="gray", linestyle="--", label="Phase 1→2")
    ax.set_title(metric)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Fine-tuning PanEcho for EF", fontsize=14)
plt.tight_layout()
plt.savefig(OUT_DIR / "training_curves.png", dpi=150)
plt.show()
print(f"Plot saved to {OUT_DIR / 'training_curves.png'}")

## Cell 14 — Evaluate on the test set

The test set was never seen during training or used to pick the best model — it's a completely held-out set that gives an unbiased measure of real-world performance.

We load the best saved checkpoint and run it on the test split.

In [ ]:
# Load the best saved model
model.load_state_dict(torch.load(OUT_DIR / "best_ef_model.pt", map_location=DEVICE))
model.eval()

test_df = filelist[filelist["Split"].str.upper() == "TEST"]
test_ds = EchoEFDataset(test_df, VIDEO_DIR, CLIP_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Test videos: {len(test_ds)}")

all_preds, all_targets = [], []
with torch.no_grad():
    for videos, efs in tqdm(test_loader, desc="test"):
        preds_dict = model(videos.to(DEVICE))
        all_preds.append(preds_dict["EF"].squeeze().cpu())
        all_targets.append(efs)

all_preds   = torch.cat(all_preds)
all_targets = torch.cat(all_targets)
test_metrics = compute_metrics(all_preds, all_targets)

print(f"\nTest set results:")
print(f"  MAE:  {test_metrics['MAE']:.2f}%")
print(f"  RMSE: {test_metrics['RMSE']:.2f}%")
print(f"  R²:   {test_metrics['R2']:.3f}")

## Cell 15 — Scatter plot: predicted vs actual EF

A scatter plot of predicted EF vs ground truth EF is the clearest way to see model quality. Points on the diagonal = perfect predictions. Spread around the diagonal = error.

In [ ]:
preds_np   = all_preds.numpy()
targets_np = all_targets.numpy()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(targets_np, preds_np, alpha=0.3, s=10, color="steelblue")
ax.plot([0, 100], [0, 100], "r--", linewidth=1, label="Perfect prediction")
ax.set_xlabel("Ground Truth EF (%)")
ax.set_ylabel("Predicted EF (%)")
ax.set_title(f"PanEcho fine-tuned EF — Test set\nMAE={test_metrics['MAE']:.2f}%  R²={test_metrics['R2']:.3f}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "ef_scatter.png", dpi=150)
plt.show()
print(f"Scatter plot saved to {OUT_DIR / 'ef_scatter.png'}")

## Cell 16 — Before vs After comparison

Side-by-side comparison of the pretrained weights vs the fine-tuned weights.
This is the payoff — how much did fine-tuning actually improve EF prediction?

In [ ]:
labels  = ["Pretrained\n(standard weights)", "Fine-tuned\n(new weights)"]
mae_vals  = [baseline_metrics["MAE"],  test_metrics["MAE"]]
rmse_vals = [baseline_metrics["RMSE"], test_metrics["RMSE"]]
r2_vals   = [baseline_metrics["R2"],   test_metrics["R2"]]

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
colors = ["#d9534f", "#5cb85c"]   # red = baseline, green = fine-tuned

for ax, vals, title, lower_better in zip(
    axes,
    [mae_vals, rmse_vals, r2_vals],
    ["MAE (%) — lower is better", "RMSE (%) — lower is better", "R² — higher is better"],
    [True, True, False]
):
    bars = ax.bar(labels, vals, color=colors, width=0.4, edgecolor="white")
    ax.set_title(title, fontsize=11)
    ax.set_ylim(0, max(vals) * 1.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(vals) * 0.03,
                f"{val:.2f}", ha="center", va="bottom", fontweight="bold")

# Print a text summary too
mae_improvement = baseline_metrics["MAE"] - test_metrics["MAE"]
print(f"EF MAE improvement:  {baseline_metrics['MAE']:.2f}% → {test_metrics['MAE']:.2f}%  "
      f"({mae_improvement:+.2f} percentage points)")
print(f"R² improvement:      {baseline_metrics['R2']:.3f} → {test_metrics['R2']:.3f}")

plt.suptitle("PanEcho EF prediction: standard vs fine-tuned weights", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / "before_after_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {OUT_DIR / 'before_after_comparison.png'}")